In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

START = "2025-04-01"      # risk index reliable from here
WAR_START = "2026-02-28"

# Point-in-time features (geopolitics, inventories, positioning, growth, weather, gas)
f = spark.table("oil_shock.gold.fct_model_features_daily").select(
    "price_date", "brent_ret_1d",
    "gulf_risk_surprise_z_3d_max", "redsea_risk_surprise_z_3d_max", "russia_risk_surprise_z_3d_max",
    "gulf_talks_z_3d_max", "crude_stocks_commercial_kbbl", "brent_mm_net_pct_oi", "wti_mm_net_pct_oi",
    "china_cli", "us_cli", "us_industrial_production_yoy_pct", "storms_in_gulf", "hurricane_in_gulf",
    "henry_hub").toPandas()

# Same-day market drivers (dollar, rates, fear)
m = spark.table("oil_shock.gold.fct_macro_daily").select(
    "obs_date", "usd_index", "treasury_10y_yield", "fed_funds_rate", "vix_model_only").toPandas()

# Same-day chokepoint transits (market sees these via tanker tracking)
c = spark.sql("""
    SELECT transit_date,
           MAX(CASE WHEN lower(chokepoint_name) LIKE '%hormuz%' THEN ships_tanker_pct_of_2025 END) AS hormuz_pct,
           MAX(CASE WHEN lower(chokepoint_name) LIKE '%mandeb%' THEN ships_tanker_pct_of_2025 END) AS mandeb_pct,
           MAX(CASE WHEN lower(chokepoint_name) LIKE '%suez%'   THEN ships_tanker_pct_of_2025 END) AS suez_pct
    FROM oil_shock.gold.fct_chokepoint_daily
    GROUP BY transit_date
""").toPandas()

for df, col in ((f, "price_date"), (m, "obs_date"), (c, "transit_date")):
    df[col] = pd.to_datetime(df[col])

df = f[f["price_date"] >= "2025-03-01"].sort_values("price_date")
df = pd.merge_asof(df, m.sort_values("obs_date"), left_on="price_date", right_on="obs_date", direction="backward")
df = pd.merge_asof(df, c.sort_values("transit_date"), left_on="price_date", right_on="transit_date", direction="backward")

levels = ["crude_stocks_commercial_kbbl", "brent_mm_net_pct_oi", "wti_mm_net_pct_oi", "china_cli", "us_cli",
          "us_industrial_production_yoy_pct", "henry_hub", "usd_index", "treasury_10y_yield",
          "fed_funds_rate", "vix_model_only", "hormuz_pct", "mandeb_pct", "suez_pct"]
df[levels] = df[levels].ffill()

df["d_hormuz"] = df["hormuz_pct"].diff()
df["d_mandeb"] = df["mandeb_pct"].diff()
df["d_suez"] = df["suez_pct"].diff()
df["d_crude_stocks"] = df["crude_stocks_commercial_kbbl"].diff()
df["d_brent_mm"] = df["brent_mm_net_pct_oi"].diff()
df["d_wti_mm"] = df["wti_mm_net_pct_oi"].diff()
df["d_ln_usd"] = np.log(df["usd_index"]).diff()
df["d_10y"] = df["treasury_10y_yield"].diff()
df["d_fed"] = df["fed_funds_rate"].diff()
df["d_vix"] = df["vix_model_only"].diff()
df["d_china_cli"] = df["china_cli"].diff()
df["d_us_cli"] = df["us_cli"].diff()
df["d_us_ip"] = df["us_industrial_production_yoy_pct"].diff()
df["d_ln_hh"] = np.log(df["henry_hub"].where(df["henry_hub"] > 0)).diff()
df["hurricane"] = df["hurricane_in_gulf"].astype(float)

GROUPS = {
    "Geopolitics": ["gulf_risk_surprise_z_3d_max", "redsea_risk_surprise_z_3d_max",
                    "russia_risk_surprise_z_3d_max", "gulf_talks_z_3d_max"],
    "Flows": ["d_hormuz", "d_mandeb", "d_suez"],
    "Inventories": ["d_crude_stocks"],
    "Positioning": ["d_brent_mm", "d_wti_mm"],
    "Dollar and rates": ["d_ln_usd", "d_10y", "d_fed"],
    "Fear": ["d_vix"],
    "Growth": ["d_china_cli", "d_us_cli", "d_us_ip"],
    "Weather and gas": ["storms_in_gulf", "hurricane", "d_ln_hh"],
}
FEATURES = [x for g in GROUPS.values() for x in g]
FEATURE_GROUP = {x: g for g, xs in GROUPS.items() for x in xs}

df = df[(df["price_date"] >= START) & df["brent_ret_1d"].notna()].copy()
df[FEATURES] = df[FEATURES].fillna(0)
df["regime"] = np.where(df["price_date"] >= WAR_START, "war", "pre_war")
ALPHAS = np.logspace(-2, 3, 30)

fit_rows, coef_rows, daily_parts = [], [], []
for reg, d in df.groupby("regime"):
    X, y = d[FEATURES].values, d["brent_ret_1d"].values
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    model = RidgeCV(alphas=ALPHAS).fit(Xs, y)

    preds, actual = [], []
    for tr, te in TimeSeriesSplit(n_splits=5).split(Xs):
        sc = StandardScaler().fit(X[tr])
        mdl = RidgeCV(alphas=ALPHAS).fit(sc.transform(X[tr]), y[tr])
        preds.append(mdl.predict(sc.transform(X[te])))
        actual.append(y[te])

    fit_rows.append({"regime": reg, "days": len(d), "first_day": d["price_date"].min().date(),
                     "last_day": d["price_date"].max().date(), "alpha": round(model.alpha_, 3),
                     "r2_in_sample": round(model.score(Xs, y), 3),
                     "r2_held_out": round(r2_score(np.concatenate(actual), np.concatenate(preds)), 3)})

    for feat, coef in zip(FEATURES, model.coef_):
        coef_rows.append({"regime": reg, "group": FEATURE_GROUP[feat], "feature": feat,
                          "pp_per_1sd_move": round(100 * coef, 3)})

    contrib = Xs * model.coef_
    part = pd.DataFrame({"price_date": d["price_date"].values, "regime": reg,
                         "actual_pct": 100 * y, "average_day_pct": 100 * model.intercept_})
    for g, xs in GROUPS.items():
        idx = [FEATURES.index(x) for x in xs]
        part[g] = 100 * contrib[:, idx].sum(axis=1)
    part["Unexplained"] = 100 * (y - model.predict(Xs))
    daily_parts.append(part)

fit = pd.DataFrame(fit_rows)
coefs = pd.DataFrame(coef_rows)
daily = pd.concat(daily_parts, ignore_index=True).sort_values("price_date")

PARTS = list(GROUPS) + ["Unexplained"]
daily["week_start"] = daily["price_date"] - pd.to_timedelta(daily["price_date"].dt.dayofweek, unit="D")
weekly = daily.groupby(["week_start", "regime"])[["actual_pct", "average_day_pct"] + PARTS].sum().reset_index()

share = (weekly.groupby("regime")[PARTS].apply(lambda x: x.abs().mean())
         .apply(lambda r: 100 * r / r.sum(), axis=1).round(1).T.reset_index()
         .rename(columns={"index": "driver"}))

print("Fit (R² = share of daily Brent moves explained):")
display(fit)
print("Share of weekly movement by driver (%):")
display(share)
print("Last 8 weeks (percentage points of Brent's weekly move):")
display(weekly.tail(8).round(2))
print("Coefficients (pp of daily Brent move per 1 standard-deviation driver move):")
display(coefs.sort_values(["regime", "group"]))

In [0]:
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

NOTES = {
    "Geopolitics": "News-based risk and talks signals (GDELT). Association, not proof of cause.",
    "Flows": "Daily tanker transits vs 2025 normal (IMF PortWatch).",
    "Inventories": "EIA weekly US crude stock change; US-only signal.",
    "Positioning": "Fund net positions (CFTC, ICE). Funds can follow prices as well as lead them.",
    "Dollar and rates": "Moves with oil (two-way): in the war, the oil shock itself lifts yields. Not a cause.",
    "Fear": "VIX changes (model input only; Cboe via FRED).",
    "Growth": "OECD leading indicators (China, US) and US industrial production.",
    "Weather and gas": "Gulf of Mexico storms and Henry Hub gas.",
    "Unexplained": "Not captured by public drivers; includes noise in the physical Brent price.",
    "Average day": "Average daily drift in the period (the war-time uptrend).",
}

def to_long(wide, keys):
    long = wide.rename(columns={"average_day_pct": "Average day"}).melt(
        id_vars=keys + ["actual_pct"], value_vars=["Average day"] + PARTS,
        var_name="driver", value_name="contribution_pp")
    long["driver_note"] = long["driver"].map(NOTES)
    return long

daily_long = to_long(daily.drop(columns=["week_start"]), ["price_date", "regime"])
weekly_long = to_long(weekly, ["week_start", "regime"])
share_long = share.melt(id_vars="driver", var_name="regime", value_name="share_pct")
share_long["driver_note"] = share_long["driver"].map(NOTES)

tables = {"driver_attribution_daily": daily_long,
          "driver_attribution_weekly": weekly_long,
          "driver_attribution_share": share_long,
          "driver_attribution_fit": fit.astype({"first_day": str, "last_day": str}),
          "driver_attribution_coefs": coefs}
for name, out in tables.items():
    out = out.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-driver-attribution")
with mlflow.start_run(run_name=f"driver_attribution {run_stamp}"):
    mlflow.log_params({
        "model": "RidgeCV on standardized daily drivers, separate pre-war and war fits",
        "target": "EIA Brent physical spot daily log return",
        "start": START, "war_start": WAR_START,
        "held_out_method": "TimeSeriesSplit(5)",
        "same_day_inputs": "dollar, rates, VIX, chokepoint transits",
        "point_in_time_inputs": "risk index, EIA stocks, COT, OECD CLI, US IP, storms, Henry Hub",
        "excluded": "product cracks (derived from Brent)",
    })
    for _, r in fit.iterrows():
        mlflow.log_metrics({f"{r['regime']}_r2_in_sample": float(r["r2_in_sample"]),
                            f"{r['regime']}_r2_held_out": float(r["r2_held_out"]),
                            f"{r['regime']}_alpha": float(r["alpha"]),
                            f"{r['regime']}_days": int(r["days"])})
    for name in ["driver_attribution_share", "driver_attribution_fit", "driver_attribution_coefs"]:
        path = f"/tmp/{name}.csv"
        tables[name].to_csv(path, index=False)
        mlflow.log_artifact(path)
print("MLflow run logged.")